# 03 — ESMFold2 Candidate Screening Pipeline (Colab L4)

## Mục đích & Cửa sàng thi đấu
Notebook này tái dựng **chính xác cửa sàng chấm điểm** của cuộc thi (Changjiang Cup 2026):
1. **Cấu hình ESMFold2 đã xác minh** từ notebook 02 (`dtype=None` / fp32 trunk+heads, `esmc_precision="fp8"`, `kernel_backend="fused"`).
2. **Cửa sàng chuẩn**: 1 seed (`seed=0`), 1 sample (`num_diffusion_samples=1`), no template.
3. **Target MSAs**: Tự động nạp MSA thực tế cho chain target từ `data/msa/` (`IL7Ra` depth 921, `TL1A` depth 2037). Binder de novo **không** có MSA (`msa=None`).
4. **Cả 2 target**:
   - **IL-7Rα**: Target là monomer 219 aa (chain A, UniProt P16871 E21–D239); candidate là chain B.
   - **TL1A**: Target là trimer sinh học 3 × 180 aa (chains A, B, C, UniProt O95150 L72–L251, broadcast MSA); candidate là chain Z.
5. **Chỉ số giao diện**: Ghi PDB theo đúng token order, chạy DunbrackLab `ipsae.py` (đường AF2, PAE cutoff 10, dist cutoff 10), trích xuất `ipSAE`, `iptm`, `plddt`.
6. **Kiểm tra tính mới (Novelty check)**: So khớp Smith-Waterman (parasail) với cơ sở dữ liệu known binders (`data/known_binders/`). Luật thi: **FAIL nếu coverage ≥ 80% VÀ identity > 70%**.
7. **Xuất kết quả**: Bảng CSV xếp hạng + JSON tóm tắt lưu trực tiếp lên Google Drive (`/content/drive/MyDrive/proteinAI/out/`).

## Yêu cầu phần cứng & Chi phí dự tính
- **GPU**: L4 (≥ 20 GB VRAM). Đã xác nhận: weights ~6.95 GB, phức 640 aa (TL1A trimer + binder) chạy tốt không OOM.
- **Thời gian thực tế đo được trên L4** (20 loops / 200 steps):
  - IL-7Rα (274 aa): ~35 giây / candidate.
  - TL1A trimer (640 aa): ~102 giây / candidate.
- **Dự trù**: 10–20 candidate mất ~10–25 phút (~1–2 Compute Units trên Colab Pro).
- **Không tải lại weights**: Tận dụng cache 24.9 GB đã có trên Google Drive.

## Cell 1 — TIỀN KIỂM & NGÂN SÁCH

Kiểm tra môi trường GPU L4, VRAM (≥ 20 GB), mount Google Drive, kiểm tra cache weights và thiết lập timer trần cứng.

In [ ]:
# ============================== CELL 1: PRE-FLIGHT ==============================
import os, sys, time, json, shutil, subprocess, platform

# Bắt buộc set trước khi import torch để giảm phân mảnh bộ nhớ VRAM
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ---------------- Cấu hình chung & Knobs ----------------
HARD_CAP_MIN        = 90       # Trần cứng an toàn (90 phút)
CU_PER_HOUR         = 4.8      # L4 Colab Pro (~4.8 CU/giờ)

# Cấu hình ESMFold2 đã xác minh từ Notebook 02
MODEL_DTYPE         = None     # fp32 trunk+heads (ĐÚNG chuẩn tác giả thiết kế)
ESMC_PRECISION      = "fp8"    # ESMC fp8 tiết kiệm ~6.8 GB VRAM
KERNEL_BACKEND      = "fused"  # Fused triton kernels

# Thông số cửa sàng đánh giá
GATE_NUM_LOOPS      = 20       # 20 recycle loops
GATE_SAMPLING_STEPS = 200      # 200 diffusion steps (chuẩn thi đấu)
GATE_LM_DROPOUT     = 0.3      # 0.3 chuẩn folding-eval
MSA_MAX_DEPTH       = 1024
MSA_COLUMN_MASK_RATE = 0.1
PAE_CUTOFF, DIST_CUTOFF = 10, 10

# Chế độ chạy nhanh (dùng để test/dry-run nếu cần)
FAST_MODE           = False    # Nếu True: 10 loops, 50 steps
STAGE_TO_LOCAL      = False    # Mặc định False: đọc trực tiếp từ Drive chỉ mất ~100s, không tốn 10-12 phút copy 25GB mỗi lần reconnect
UA                  = "proteinAI-changjiangcup/0.3 (screening pipeline)"

T0 = time.time()
def elapsed_min(): return (time.time() - T0) / 60.0
def cu_burned():   return elapsed_min() / 60.0 * CU_PER_HOUR

def budget(step=""):
    m, cu = elapsed_min(), cu_burned()
    bar = "!!!" if m > 0.8 * HARD_CAP_MIN else ("!" if m > 0.5 * HARD_CAP_MIN else "")
    print(f"[NGÂN SÁCH]{bar} {m:6.1f} phút · ước {cu:4.2f} CU "
          f"· trần {HARD_CAP_MIN} phút ({HARD_CAP_MIN/60*CU_PER_HOUR:.1f} CU) {step}", flush=True)
    if m > HARD_CAP_MIN:
        try: save_partial()
        except Exception: pass
        raise SystemExit(f"DỪNG CỨNG: Đã chạy {m:.0f} phút (> {HARD_CAP_MIN} min). Báo cáo từng phần đã ghi ra Drive.")

def have_budget(need_min, what):
    left = HARD_CAP_MIN - elapsed_min()
    if left < need_min:
        print(f"BỎ QUA {what}: còn {left:.0f} phút, cần ~{need_min} phút.")
        return False
    return True

ENV = {"python": platform.python_version(), "platform": platform.platform()}

try:
    import torch
except Exception as e:
    raise SystemExit(f"DỪNG: Không import được torch ({e}).")

if not torch.cuda.is_available():
    raise SystemExit("DỪNG: KHÔNG CÓ GPU. Runtime > Change runtime type > GPU (L4).")

gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
free_b, total_b = torch.cuda.mem_get_info(0)
vram_total_gb, vram_free_gb = total_b / 1024**3, free_b / 1024**3
ENV.update(torch_version=torch.__version__, gpu=gpu_name, compute_capability=list(cc),
           vram_total_gb=round(vram_total_gb, 2), vram_free_gb=round(vram_free_gb, 2))
print(f"GPU    : {gpu_name} (cc {cc[0]}.{cc[1]})")
print(f"VRAM   : {vram_total_gb:.2f} GB tổng · {vram_free_gb:.2f} GB trống")
print(f"torch  : {torch.__version__}")

if vram_total_gb < 20.0:
    raise SystemExit(f"DỪNG: GPU '{gpu_name}' chỉ có {vram_total_gb:.1f} GB VRAM, cần ≥ 20 GB (L4) để chạy TL1A trimer.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    raise SystemExit(f"DỪNG: Không mount được Google Drive ({e}).")

ROOT    = "/content/drive/MyDrive/proteinAI"
D_W     = f"{ROOT}/weights/ESMFold2"
D_DATA  = f"{ROOT}/data"
D_MSA   = f"{D_DATA}/msa"
D_KNOWN = f"{D_DATA}/known_binders"
D_RUNS  = f"{ROOT}/runs/03_screening"
D_OUT   = f"{ROOT}/out"
D_LOGS  = f"{ROOT}/logs"
for d in (D_DATA, D_MSA, D_KNOWN, D_RUNS, D_OUT, D_LOGS):
    os.makedirs(d, exist_ok=True)
L_W = "/content/esmfold2_weights"

if not os.path.exists(f"{D_W}/config.json"):
    raise SystemExit(f"DỪNG: Không thấy cache weights tại {D_W}/config.json. Hãy chạy notebook 01/02 trước để đồng bộ weights.")

drive_free_gb   = shutil.disk_usage(ROOT).free / 1024**3
content_free_gb = shutil.disk_usage("/content").free / 1024**3
ENV.update(drive_free_gb=round(drive_free_gb, 1), content_free_gb=round(content_free_gb, 1))
print(f"Drive  : còn {drive_free_gb:.1f} GB · /content: còn {content_free_gb:.1f} GB")
if drive_free_gb < 2.0:
    raise SystemExit(f"DỪNG: Drive chỉ còn {drive_free_gb:.1f} GB, không đủ ghi kết quả screening.")
if STAGE_TO_LOCAL and content_free_gb < 30.0:
    print("CẢNH BÁO: /content < 30 GB → tắt STAGE_TO_LOCAL, đọc trực tiếp từ Drive.")
    STAGE_TO_LOCAL = False

REPORT = {
    "notebook": "03_esmfold2_screening",
    "schema": 1,
    "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "env": ENV,
    "knobs": {
        "model_dtype": str(MODEL_DTYPE), "esmc_precision": ESMC_PRECISION,
        "kernel_backend": KERNEL_BACKEND,
        "gate_num_loops": 10 if FAST_MODE else GATE_NUM_LOOPS,
        "gate_num_sampling_steps": 50 if FAST_MODE else GATE_SAMPLING_STEPS,
        "gate_lm_dropout": GATE_LM_DROPOUT, "fast_mode": FAST_MODE,
        "pae_cutoff": PAE_CUTOFF, "dist_cutoff": DIST_CUTOFF,
    },
    "steps": {}, "screened_candidates": [], "errors": []
}
PARTIAL = f"{D_OUT}/03_partial.json"

def save_partial():
    REPORT["elapsed_min"] = round(elapsed_min(), 2)
    REPORT["cu_estimate"] = round(cu_burned(), 3)
    tmp = PARTIAL + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(REPORT, f, indent=2, default=str)
    os.replace(tmp, PARTIAL)

save_partial()
print(f"\nBáo cáo từng phần sẽ ghi vào: {PARTIAL}")
print("TIỀN KIỂM: HOÀN TẤT.")
budget("(cell 1 xong)")

## Cell 2 — CÀI ĐẶT THƯ VIỆN

Cài đặt `esm==3.4.3` và `parasail` (dùng cho Smith-Waterman novelty check). Kiểm tra phiên bản torch không bị thay đổi.

In [ ]:
# ============================== CELL 2: INSTALL PACKAGES ========================
budget("(bắt đầu cell 2)")
t = time.time()
torch_before = torch.__version__

packages_to_install = []
try:
    import esm as _probe
    if getattr(_probe, "__version__", "") != "3.4.3":
        packages_to_install.append("esm==3.4.3")
except Exception:
    packages_to_install.append("esm==3.4.3")

try:
    import parasail  # noqa
except Exception:
    packages_to_install.append("parasail")

if packages_to_install:
    print(f"Cài đặt: {packages_to_install} ...", flush=True)
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages_to_install
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        REPORT["errors"].append({"step": "install", "stderr": r.stderr[-2000:]})
        save_partial()
        raise SystemExit(f"DỪNG: Lỗi cài đặt gói ({r.returncode}).")
else:
    print("esm 3.4.3 và parasail đã sẵn sàng — bỏ qua pip.")

import importlib
importlib.invalidate_caches()
import torch as _t2
torch_after = _t2.__version__
if torch_after != torch_before:
    raise SystemExit(f"DỪNG: pip đã thay đổi torch {torch_before} -> {torch_after}. Hãy Restart Session.")

from esm.models.esmfold2 import (
    EsmFold2Model, ESMFold2InputBuilder, ProteinInput, StructurePredictionInput, MSA,
)
from esm.utils.msa.msa import is_a3m_insertion
import esm as _esm
import parasail

install_s = time.time() - t
print(f"\nesm {getattr(_esm, '__version__', '?')} · parasail {getattr(parasail, '__version__', '?')} · torch {torch_after} (giữ nguyên) [{install_s:.1f}s]")
REPORT["steps"]["install"] = {"esm_version": getattr(_esm, "__version__", None), "seconds": round(install_s, 1)}
save_partial()
budget("(cell 2 xong)")

## Cell 3 — XÁC MINH & STAGE WEIGHTS (Tải 0 byte)

Kiểm tra tính toàn vẹn từng file checkpoint ESMFold2 từ Drive và copy sang bộ nhớ cục bộ `/content/esmfold2_weights` để đạt tốc độ nạp cao nhất.

In [ ]:
# ============================== CELL 3: VERIFY & STAGE WEIGHTS ==================
budget("(bắt đầu cell 3)")

MANIFEST = {
    "config.json":                      8214,
    "model.safetensors.index.json":     245594,
    "model-00001-of-00006.safetensors": 4955672788,
    "model-00002-of-00006.safetensors": 4971194264,
    "model-00003-of-00006.safetensors": 4997460576,
    "model-00004-of-00006.safetensors": 4979069192,
    "model-00005-of-00006.safetensors": 4934556472,
    "model-00006-of-00006.safetensors": 1481396616,
    "ccd.pkl":                          417306584,
}
MANIFEST_TOTAL = sum(MANIFEST.values())

def audit_weights(d):
    ok, bad = [], []
    for fn, size in MANIFEST.items():
        p = os.path.join(d, fn)
        if not os.path.exists(p):
            bad.append((fn, "missing", 0, size))
        else:
            got = os.path.getsize(p)
            (ok if got == size else bad).append((fn, "ok" if got == size else "size-mismatch", got, size))
    return ok, bad

ok, bad = audit_weights(D_W)
print(f"Cache Drive {D_W}: {len(ok)}/{len(MANIFEST)} files đúng kích thước ({MANIFEST_TOTAL/1024**3:.2f} GB).")
if bad:
    for fn, why, got, want in bad:
        print(f"  LỖI {fn:38s} {why:14s} got={got} want={want}")
    raise SystemExit("DỪNG: Cache weights trên Drive bị thiếu hoặc sai lệch byte.")

MODEL_DIR = D_W
stage_s = 0.0
if STAGE_TO_LOCAL:
    os.makedirs(L_W, exist_ok=True)
    _, bad_l = audit_weights(L_W)
    if bad_l:
        print(f"Copy Drive -> {L_W} (~2-4 phút) ...", flush=True)
        t0 = time.time()
        for fn, why, got, want in bad_l:
            print(f"  copying {fn} ...", flush=True)
            shutil.copy2(os.path.join(D_W, fn), os.path.join(L_W, fn))
        stage_s = time.time() - t0
        print(f"  Xong trong {stage_s:.0f}s.")
    else:
        print(f"{L_W} đã có đủ weights sẵn từ phiên trước.")
    MODEL_DIR = L_W

CCD_DIR = MODEL_DIR
print(f"\nMODEL_DIR sử dụng: {MODEL_DIR} (Tải 0 byte từ internet)")
REPORT["steps"]["weights"] = {"model_dir": MODEL_DIR, "stage_seconds": round(stage_s, 1)}
save_partial()
budget("(cell 3 xong)")

## Cell 4 — TARGET DEFINITIONS & NẠP MSA PRE-CACHED

- **IL-7Rα**: UniProt `P16871` Glu21–Asp239 = 219 aa.
- **TL1A**: UniProt `O95150` Leu72–Leu251 = 180 aa / protomer (trimer = 540 aa).
- **Nạp MSA thực tế** từ `data/msa/`:
  - `IL7Ra_P16871_E21-D239.uniref.a3m` (depth 921)
  - `TL1A_O95150_L72-L251.uniref.a3m` (depth 2037)
- Xác nhận match columns và query sequence khớp 100% để đảm bảo `prepare_input` không lệch token.

In [ ]:
# ============================== CELL 4: TARGETS & MSAS ==========================
budget("(bắt đầu cell 4)")
import hashlib

# Chuỗi chuẩn theo UniProt quy định trong quy chế thi đấu
IL7RA_SEQ = (
    "ESGYAQNGDLEDAELDDYSFSCYSQLEVNGSQHSLTCAFEDPDVNITNLEFEICGALVEVKCLNFRKLQEIYFIETK"
    "KFLLIGKSNICVKVGEKSLTCKKIDLTTIVKPEAPFDLSVVYREGANDFVVTFNTSHLQKKYVKVLMHDVAYRQEKD"
    "ENKWTHVNLSSTKLTLLQRKLQPAAMYEIKVRSIPDHYFKGFWSEWSPSYYFRTPEINNSSGEMD"
)
TL1A_SEQ = (
    "LKGQEFAPSHQQVYAPLRADGDKPRAHLTVVRQTPTQHFKNQFPALHWEHELGLAFTKNRMNYTNKFLLIPESGDYF"
    "IYSQVTFRGMTSECSEIRQAGRPNKPDSITVVITKVTDSYPEPTQLLMGTKSVCEVGSNWFQPIYLGAMFSLQEGDK"
    "LMVNVSDISLVDYTKEDKTFFGAFLL"
)

assert len(IL7RA_SEQ) == 219, f"IL7RA sequence length {len(IL7RA_SEQ)} != 219"
assert len(TL1A_SEQ) == 180, f"TL1A sequence length {len(TL1A_SEQ)} != 180"

TARGET_METAS = {
    "IL7RA": {"name": "IL7Ra_P16871_E21-D239", "seq": IL7RA_SEQ, "len": 219,
              "sha256": "08e2a05b5b4787bb28c03302b1d56c3856980423a5582f2096bc6f5238b0be03"},
    "TL1A":  {"name": "TL1A_O95150_L72-L251",  "seq": TL1A_SEQ,  "len": 180,
              "sha256": "4464ad15470e2ecda72ea879beb5972059062085b04ff515cef26f642e092c27"},
}

def verify_and_load_a3m(target_key):
    meta = TARGET_METAS[target_key]
    path = os.path.join(D_MSA, f"{meta['name']}.uniref.a3m")
    
    # Nếu chưa có trên Drive msa, kiểm tra thư mục repo cục bộ /content
    if not os.path.exists(path):
        alt_path = f"/content/proteinAI/data/msa/{meta['name']}.uniref.a3m"
        if os.path.exists(alt_path):
            shutil.copy2(alt_path, path)
    
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy MSA file: {path}. Đảm bảo data/msa/ đã được đồng bộ lên Drive.")
    
    raw = open(path, "rb").read()
    lines = [l for l in raw.decode("utf8", "replace").split("\n") if l and not l.startswith("#")]
    hdr = [i for i, l in enumerate(lines) if l.startswith(">")]
    q = lines[hdr[0] + 1] if hdr else ""
    match_cols = sum(1 for c in q if not is_a3m_insertion(c))
    plain_q = "".join(c for c in q if not is_a3m_insertion(c)).replace("-", "")
    
    assert plain_q == meta["seq"], f"{target_key}: Query trong MSA không khớp target sequence!"
    assert match_cols == meta["len"], f"{target_key}: Số cột match ({match_cols}) != độ dài ({meta['len']})!"
    
    msa_obj = MSA.from_a3m(path=path)
    print(f"Nạp MSA [{target_key:5s}]: depth = {msa_obj.depth:4d} · match_columns = {match_cols:3d} (OK)")
    return msa_obj

MSA_IL7RA = verify_and_load_a3m("IL7RA")
MSA_TL1A  = verify_and_load_a3m("TL1A")

REPORT["steps"]["msa"] = {
    "il7ra_depth": MSA_IL7RA.depth,
    "tl1a_depth":  MSA_TL1A.depth,
}
save_partial()
budget("(cell 4 xong)")

## Cell 5 — NẠP ESMFold2 MODEL

Khởi tạo `EsmFold2Model` với cấu hình fp32 trunk+heads, ESMC fp8 và backend fused triton kernel. Khởi tạo `ESMFold2InputBuilder` từ cache `ccd.pkl`.

In [ ]:
# ============================== CELL 5: LOAD MODEL ==============================
budget("(bắt đầu cell 5)")
import gc

gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t_load = time.time()

print(f"Nạp ESMFold2 (dtype=None/fp32, esmc_precision={ESMC_PRECISION}) ... (~100s)", flush=True)
model = EsmFold2Model.from_pretrained(
    MODEL_DIR, device="cuda", dtype=MODEL_DTYPE, esmc_precision=ESMC_PRECISION, load_esmc=True,
).eval()

try:
    model.set_kernel_backend(KERNEL_BACKEND)
    print(f"Kernel backend: {KERNEL_BACKEND} (thành công)")
except Exception as e:
    model.set_kernel_backend(None)
    print(f"CẢNH BÁO: Không bật được kernel '{KERNEL_BACKEND}' ({e}) -> dùng reference path")

t_builder = time.time()
builder = ESMFold2InputBuilder(ccd_cache=CCD_DIR)
load_total_s = time.time() - t_load

vram_alloc = torch.cuda.memory_allocated() / 1024**3
vram_free  = torch.cuda.mem_get_info(0)[0] / 1024**3
print(f"\nModel nạp xong trong {load_total_s:.1f}s · VRAM weights: {vram_alloc:.2f} GB · VRAM trống: {vram_free:.2f} GB")
print(f"Số tham số: {sum(p.numel() for p in model.parameters()):,} · buffers: {sum(b.numel() for b in model.buffers()):,}")

REPORT["steps"]["load_model"] = {
    "seconds": round(load_total_s, 1),
    "vram_weights_gb": round(vram_alloc, 2),
    "vram_free_after_load_gb": round(vram_free, 2),
    "msa_encoder_present": model.msa_encoder is not None,
}
save_partial()
budget("(cell 5 xong)")

## Cell 6 — ADAPTER ipSAE & GHI PDB THEO TOKEN ORDER

Chuẩn bị công cụ tính `ipSAE` (DunbrackLab `ipsae.py`). Ghi file PDB theo **đúng thứ tự token của ESMFold2** đảm bảo token permutation là đồng nhất tuyệt đối (`np.arange(N)`).

In [ ]:
# ============================== CELL 6: ipSAE ADAPTER ==========================
budget("(bắt đầu cell 6)")
import numpy as np, urllib.request

IPSAE_PY = f"{D_DATA}/ipsae.py"
IPSAE_URL = "https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py"

if not os.path.exists(IPSAE_PY) or os.path.getsize(IPSAE_PY) < 5000:
    print(f"Tải ipsae.py từ {IPSAE_URL} ...")
    req = urllib.request.Request(IPSAE_URL, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=60) as r, open(IPSAE_PY, "wb") as f:
        f.write(r.read())

ipsae_sha = hashlib.sha256(open(IPSAE_PY, "rb").read()).hexdigest()
print(f"ipsae.py tại: {IPSAE_PY} (sha256: {ipsae_sha[:16]}...)")

def write_pdb_token_order(mc, path):
    """Ghi PDB theo đúng token order của ESMFold2. Gán B-factor = pLDDT * 100."""
    chain_lookup = mc.metadata.chain_lookup
    lines, serial, n_ca, n_cb, res_counter = [], 1, 0, 0, {}
    for tok_idx, (s, e) in enumerate(mc.token_to_atoms):
        resname = str(mc.sequence[tok_idx])[:3]
        chain = str(chain_lookup[int(mc.chain_id[tok_idx])])[:1] or "A"
        res_counter[chain] = res_counter.get(chain, 0) + 1
        resnum = res_counter[chain]
        b = float(mc.plddt[tok_idx]) * 100.0
        has_ca = has_cb = False
        for a in range(int(s), int(e)):
            an = str(mc.atom_names[a]).strip().upper()
            el = str(mc.atom_elements[a]).strip().upper()[:2]
            x, y, z = (float(v) for v in mc.atom_positions[a])
            an4 = f" {an:<3}" if len(an) < 4 else an[:4]
            lines.append(f"ATOM  {serial:5d} {an4} {resname:>3s} {chain:1s}{resnum:4d}    "
                         f"{x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{b:6.2f}          {el:>2s}")
            serial += 1
            if an == "CA":
                has_ca = True
                if resname == "GLY": has_cb = True
            if an == "CB":
                has_cb = True
        n_ca += has_ca; n_cb += has_cb
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines + ["TER", "END"]) + "\n")
    return len(mc.token_to_atoms), n_ca, n_cb

def run_ipsae_eval(result, tag, outdir, binder_chain):
    """Chạy ipsae.py theo đường AF2 (.pdb + .json). Trích xuất điểm ipSAE tương tác với binder_chain."""
    os.makedirs(outdir, exist_ok=True)
    pdb_p  = f"{outdir}/{tag}.pdb"
    json_p = f"{outdir}/{tag}.json"
    n_res, n_ca, n_cb = write_pdb_token_order(result.complex, pdb_p)

    pae = result.pae.detach().cpu().float().numpy()
    plddt = result.plddt.detach().cpu().float().numpy().ravel()
    if plddt.max() <= 1.0:
        plddt = plddt * 100.0

    assert pae.shape == (n_res, n_res), f"PAE shape {pae.shape} != ({n_res}, {n_res})"
    assert n_ca == n_res, f"Số CA ({n_ca}) != số residue ({n_res})"
    assert n_cb == n_res, f"Số CB ({n_cb}) != số residue ({n_res})"

    with open(json_p, "w", encoding="utf-8") as f:
        json.dump({"pae": np.round(pae, 2).tolist(), "plddt": np.round(plddt, 2).tolist(),
                   "iptm": float(result.iptm) if result.iptm is not None else -1.0,
                   "ptm": float(result.ptm) if result.ptm is not None else -1.0}, f)

    r = subprocess.run([sys.executable, IPSAE_PY, json_p, pdb_p, str(PAE_CUTOFF), str(DIST_CUTOFF)],
                       capture_output=True, text=True, timeout=1800)
    txt_p = f"{outdir}/{tag}_{PAE_CUTOFF:02d}_{DIST_CUTOFF:02d}.txt"
    out = {"returncode": r.returncode, "pdb": pdb_p, "json": json_p, "txt": txt_p, "asym": {}, "max": {}}
    
    if r.returncode != 0 or not os.path.exists(txt_p):
        out["stderr"] = r.stderr[-2000:]
        return out
        
    for line in open(txt_p, encoding="utf-8"):
        p = line.split()
        if len(p) < 6 or p[0] == "Chn1": continue
        try: val = float(p[5])
        except ValueError: continue
        if p[4] == "asym":
            out["asym"][f"{p[0]}->{p[1]}"] = val
        elif p[4] == "max":
            out["max"][f"{p[0]}-{p[1]}"] = val

    # Điểm ipSAE tốt nhất của cặp có chứa chuỗi binder
    binder_asym = {k: v for k, v in out["asym"].items() if binder_chain in k}
    out["ipsae_binder_best"] = max(binder_asym.values()) if binder_asym else 0.0
    out["ipsae_min"] = min(out["asym"].values()) if out["asym"] else 0.0
    out["binder_asym_pairs"] = binder_asym
    return out

print("Adapter ipSAE đã sẵn sàng (đường AF2, cutoffs: 10/10).")
REPORT["steps"]["ipsae_adapter"] = {"ipsae_py": IPSAE_PY, "status": "ready"}
save_partial()
budget("(cell 6 xong)")

## Cell 7 — MODULE NOVELTY CHECK (PARASAIL SMITH-WATERMAN)

Kiểm tra tính mới dựa trên Smith-Waterman cục bộ (BLOSUM62, Gap Open 11, Gap Extend 1).
- Ngưỡng thi đấu: **FAIL nếu `coverage_ours ≥ 80%` VÀ `identity > 70%`**.
- So khớp với cơ sở dữ liệu `data/known_binders/` theo từng target.

In [ ]:
# ============================== CELL 7: NOVELTY CHECK MODULE ====================
budget("(bắt đầu cell 7)")

KNOWN_BINDER_FILES = {
    "il7ra": [
        ("IL7Ra_denovo_minibinder_designs_IPD_Cao2022.fasta", "IPD_Cao2022_122k"),
        ("IL7Ra_denovo_minibinders_VALIDATED.fasta",          "VALIDATED"),
        ("IL7Ra_therapeutic_antibodies.fasta",                "therapeutics"),
        ("IL7Ra_PDB_antibody_Fabs.fasta",                     "PDB_Fabs"),
    ],
    "tl1a": [
        ("TL1A_therapeutic_antibodies.fasta", "therapeutics"),
        ("TL1A_natural_binder_DcR3.fasta",    "DcR3_natural"),
    ],
}

def parse_fasta_records(path):
    records, header, seq_parts = [], None, []
    with open(path, "r", encoding="utf-8") as fh:
        for raw in fh:
            line = raw.strip()
            if not line: continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_parts).upper()))
                header = line[1:].strip()
                seq_parts = []
            else:
                seq_parts.append(line)
    if header is not None:
        records.append((header, "".join(seq_parts).upper()))
    return records

CORE_KNOWN_BINDERS = {
    "tl1a": [
        ("DcR3_TNFRSF6B_res30_205", "VAETPTYPWRDAETGERLVCAQCPPGTFVQRPCRRDSPTTCGPCPPRHYTQFWNYLERCRYCNVLCGEREEEARACHATHNRACRCRTGFFAHAGFCLEHASCPPGAGVIAPGTPSQNTQCQPCPPGTFSASSSSSEQCQPHRNCTALGLALNVPGSSSHDTLCTSCTGFPLSTRV"),
        ("Afimkibart_VH", "QVQLVQSGAEVKKPGASVKVSCKASGYDFTYYGISWVRQAPGQGLEWMGWISTYNGNTHYARMLQGRVTMTTDTSTRTAYMELRSLRSDDTAVYYCARENYYGSGAYRGGMDVWGQGTTVTVSS"),
        ("Afimkibart_VL", "EIVLTQSPATLSLSPGERATLSCRASQSVSSYLAWYQQKPGQAPRLLIYDASNRATGIPARFSGSGSGTDFTLTISSLEPEDFAVYYCQQRSNWPWTFGQGTKVEIK"),
        ("Duvakitug_VH", "QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYDINWVRQAPGQGLEWMGWLNPNSGYTGYAQKFQGRVTMTADRSTSTAYMELSSLRSEDTAVYYCAREVPETAAFEYWGQGTLVTVSS"),
        ("Duvakitug_VL", "QSVLTQPPSVSGAPGQRVTISCTSSSSDIGAGLGVHWYQQLPGTAPKLLIEGYYNRPSGVPDDRFSGSKSGTSASLTITGLLPEDEGDYYCQSWDGTLSALFGGGTKLTVL"),
        ("Tulisokibart_VH", "QVQLVQSGAEVKKPGASVKVSCKASGFDIQDTYMHWVKQRPGQGLEWMGRIDPASGHTKYDPKFQVRVTITRDTSTSTVYLELSSLRSEDTAVYYCARSGGLPDVWGQGTTVTVSS"),
        ("Tulisokibart_VL", "EIVLTQSPGTLSLSPGERATLSCRASSSVSYMYWYQQKPGQAPRPLIYATSNLASGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQWEGNPRTFGGGTKLEIK"),
    ],
    "il7ra": [
        ("7S5B_minibinder_optimized", "SVIEKLRKLEKQARKQGDEVLVMLARMVLEYLEKGWVSEEDADESADRIEEVLKK"),
        ("7S5B_minibinder_original", "SVKKKVRKVEKKARKAGDELAVLLARRVLEALEKGLVSEEDADESADRIEEALKK"),
        ("Bempikibart_VH", "EVQLVESGGGLVQPGRSLRLSCAASGFTFDDHAMHWVRQAPGKGLEWVSGISWNSRGIGYADSVKGRFTIFRDNAKNSLYLQMNSLRAEDTALYYCAKDEYSRGYYVLDVWGQGTTVTVSS"),
        ("Bempikibart_VL", "AIQLTQSPSSLSASVGDRVTITCRASQGISSALAWYQQKPGKAPKLLIYDASSLESGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQFNSYPLWITFGQGTRLEIK"),
        ("Crebankitug_VH", "EVQLVESGGGLVKPGGSLRLSCAASGFTFDDSVMHWVRQAPGKGLEWVSLVGWDGFFTYYADSVKGRFTISRDNAKNSLYLQMNSLRAEDTAVYYCARQGDYMGNNWGQGTLVTVSS"),
        ("Crebankitug_VL", "NFMLTQPHSVSESPGKTVTISCTRSSGSIDSSYVQWYQQRPGSSPTTVIYEDDQRPSGVPDRFSGSIDSSSNSASLTISGLKTEDEADYYCQSYDFHHLVFGGGTKLTVL"),
        ("Lusvertikimab_VH", "QVQLVESGGGLVKPGGSLRLSCAVSGFTLSDYYMAWIRQAPGKGLEWVSTISASGLRTYYPDSVKGRFTISRDNAKNSLYLQMNSLRAEDTAVYYCARPLSAHYGFNYFDYWGQGTLVTVSS"),
        ("Lusvertikimab_VL", "DIQMTQSPSSLSASVGDRVTITCRTSEDIYQGLAWYQQKPGKAPKLLLYSANTLHIGVPSRFSGSGSGTDYTLTISSLQPEDFATYYCQQYYDYPLAFGGGTKVEIK"),
        ("Fab_4A10_VH", "QVQLQQPGAELVMPGASVKLSCKASGYTFTSYWMHWVKQRPGEGLEWIGEIDPSDSYTNDNQKFKGKATLTVDKSSSTAYMQLSSLTSEDSAVYYCARRLYSNSYYYAMDYWGQGTSVTVSS"),
        ("Fab_4A10_VL", "DIQMTQSPSSLSASLGGKVTITCKASQDIKKYIAWYQHKPGKGPRLLIHYTSTLQPGIPSRFSGSGSGRDYSFSISNLEPVDIATYYCLQYDNLLTFGAGTKLELKRADAAPTVSIFPPS"),
        ("Fab_2B8_VH", "EVQLQQSGPELVKPGASVKMSCKASGYTFSDYYMHWVKQSHGKSLEWIGYIYPDNGGNGYNQKFKGKATLTVDKSSSTVYMELRSLTSEDSALYYCARGTYYDGSYFDYWGQGTTLTVSS"),
        ("Fab_2B8_VL", "DIVMTQSHKFMSTLVGDRVSITCKASQDVSTTVAWYQQKPGQSPKLLIYSASYRYTGVPDRFTGSGSGTDFTFTISSVQAEDLAVYYCQQHYSIPRTFGGGTKLEIK"),
    ],
}

def load_known_binders_db(target_name):
    t_key = target_name.lower()
    files = KNOWN_BINDER_FILES.get(t_key, [])
    records = []
    for fn, label in files:
        p = os.path.join(D_KNOWN, fn)
        if not os.path.exists(p):
            alt = f"/content/proteinAI/data/known_binders/{fn}"
            if os.path.exists(alt): p = alt
        if os.path.exists(p):
            rec = parse_fasta_records(p)
            records.extend(rec)
            print(f"  [{t_key}] Đã nạp {len(rec):>6,} seqs từ {fn} ({label})")
        else:
            print(f"  [CẢNH BÁO] Không thấy file known binders: {fn}")
    if not records:
        records = CORE_KNOWN_BINDERS.get(t_key, [])
        print(f"  [{t_key}] Dùng DB đối chứng nhúng sẵn ({len(records)} sequences chuẩn: natural + therapeutics)")
    return records

def sw_align_parasail(query, ref):
    """Smith-Waterman cục bộ qua parasail: BLOSUM62, Gap open 11, Gap extend 1."""
    res = parasail.sw_trace_striped_16(query, ref, 11, 1, parasail.blosum62)
    tb = res.traceback
    q_al, r_al = tb.query, tb.ref
    aln_cols = len(q_al)
    n_matches = sum(1 for a, b in zip(q_al, r_al) if a == b and a != '-')
    q_aligned = sum(1 for a in q_al if a != '-')
    r_aligned = sum(1 for b in r_al if b != '-')
    return n_matches, q_aligned, r_aligned, aln_cols

def evaluate_novelty(cand_name, cand_seq, known_db):
    """Kiểm tra 1 candidate với DB known binders. Trả về dict kết quả tính mới."""
    cand_len = len(cand_seq)
    if not known_db:
        return {"verdict": "PASS", "max_identity_pct": 0.0, "coverage_ours_pct": 0.0,
                "worst_hit_id": "NO_DB", "verdict_strict": "PASS"}

    best_identity, best_cov_ours, best_cov_theirs, best_hit = 0.0, 0.0, 0.0, ""
    strict_best_id, strict_best_cov = 0.0, 0.0

    for kb_name, kb_seq in known_db:
        if not kb_seq: continue
        try:
            n_mat, q_aln, r_aln, aln_cols = sw_align_parasail(cand_seq, kb_seq)
        except Exception:
            continue
        if aln_cols == 0: continue

        cov_ours   = (q_aln / cand_len) * 100.0
        cov_theirs = (r_aln / len(kb_seq)) * 100.0
        cov_strict = max(cov_ours, cov_theirs)
        identity   = (n_mat / aln_cols) * 100.0

        if cov_ours >= 80.0:
            if identity > best_identity:
                best_identity   = identity
                best_cov_ours   = cov_ours
                best_cov_theirs = cov_theirs
                best_hit        = kb_name
        
        if cov_strict >= 80.0:
            if identity > strict_best_id:
                strict_best_id  = identity
                strict_best_cov = cov_strict

    verdict = "FAIL" if best_identity > 70.0 else "PASS"
    verdict_strict = "FAIL" if strict_best_id > 70.0 else "PASS"
    return {
        "verdict": verdict,
        "verdict_strict": verdict_strict,
        "max_identity_pct": round(best_identity, 2),
        "coverage_ours_pct": round(best_cov_ours, 2),
        "coverage_theirs_pct": round(best_cov_theirs, 2),
        "worst_hit_id": best_hit,
    }

# Nạp trước DB cho cả hai target
print("Nạp cơ sở dữ liệu Known Binders:")
KB_DB = {
    "IL7RA": load_known_binders_db("il7ra"),
    "TL1A":  load_known_binders_db("tl1a"),
}
print(f"\nNovelty Checker sẵn sàng (IL-7Ra: {len(KB_DB['IL7RA']):,} seqs, TL1A: {len(KB_DB['TL1A']):,} seqs).")
REPORT["steps"]["novelty_db"] = {"il7ra_seqs": len(KB_DB["IL7RA"]), "tl1a_seqs": len(KB_DB["TL1A"])}
save_partial()
budget("(cell 7 xong)")

## Cell 8 — TIẾP NHẬN CANDIDATES & CẤU HÌNH SÀNG LỌC

Hỗ trợ tiếp nhận candidates từ file CSV / FASTA trên Drive (`INPUT_CANDIDATES_PATH`).
Nếu chưa có file đầu vào được chỉ định, notebook tự động cung cấp **tập đối chứng chuẩn** (gồm Positive control 7S5B và Negative control scrambled) để chạy kiểm chứng pipeline ngay lập tức.

In [ ]:
# ============================== CELL 8: CANDIDATE INGESTION ====================
budget("(bắt đầu cell 8)")
import csv

# ---------------- Đường dẫn file candidates đầu vào (tuỳ chọn) ----------------
# Đặt đường dẫn file .csv hoặc .fasta nếu bạn có sẵn danh sách thiết kế từ agent designer:
# Ví dụ: INPUT_CANDIDATES_PATH = f"{D_DATA}/candidates.csv"
INPUT_CANDIDATES_PATH = None   # None = dùng tập benchmark đối chứng mẫu dưới đây

# Target mặc định khi chạy candidates (chọn 'IL7RA', 'TL1A', hoặc 'PER_ROW' nếu file CSV có cột target)
DEFAULT_TARGET = "IL7RA"

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def parse_candidates_file(path):
    cand_list = []
    ext = os.path.splitext(path)[1].lower()
    if ext in (".fa", ".fasta"):
        for h, s in parse_fasta_records(path):
            cand_list.append({"name": h.split()[0], "sequence": s, "target": DEFAULT_TARGET})
    elif ext in (".csv", ".tsv"):
        delim = "\t" if ext == ".tsv" else ","
        with open(path, "r", encoding="utf-8") as fh:
            reader = csv.DictReader(fh, delimiter=delim)
            fields_low = {f.lower().strip(): f for f in reader.fieldnames or []}
            name_col = next((fields_low[c] for c in ("name", "id", "seq_id", "candidate_id") if c in fields_low), None)
            seq_col  = next((fields_low[c] for c in ("sequence", "seq", "aa") if c in fields_low), None)
            tgt_col  = next((fields_low[c] for c in ("target", "target_name") if c in fields_low), None)
            if not (name_col and seq_col):
                raise ValueError(f"CSV cần có cột name và sequence. Tìm thấy: {reader.fieldnames}")
            for row in reader:
                nm = row[name_col].strip()
                sq = row[seq_col].strip().upper()
                tg = row[tgt_col].strip().upper() if tgt_col and row.get(tgt_col) else DEFAULT_TARGET
                if nm and sq:
                    cand_list.append({"name": nm, "sequence": sq, "target": tg})
    return cand_list

# Danh sách candidates dự phòng (Benchmark Controls)
SAMPLE_BENCHMARK_CANDIDATES = [
    {
        "name": "ctrl_pos_7S5B_IL7Ra",
        "sequence": "SVIEKLRKLEKQARKQGDEVLVMLARMVLEYLEKGWVSEEDADESADRIEEVLKK",
        "target": "IL7RA",
        "note": "Positive control (Cao et al. Nature 2022, PDB 7S5B D-F, picomolar)"
    },
    {
        "name": "ctrl_neg_scrambled_IL7Ra",
        "sequence": "EVELVRLLRAKELVSEDWLSGEKRILEMKEDVLSKAYEAEKQVIDEMVAKRSDLE",
        "target": "IL7RA",
        "note": "Negative control (xáo trộn amino acid của 7S5B, mất cấu trúc)"
    },
    {
        "name": "probe_dcr3_TL1A",
        "sequence": (
            "VAETPTYPWRDAETGERLVCAQCPPGTFVQRPCRRDSPTTCGPCPPRHYTQFWNYLERCRYCNVLCGEREEEARACH"
            "ATHNRACRCRTGFFAHAGFCLEHASCPPGAGVIAPGTPSQNTQCQPCPPGTFSASSSSSEQCQPHRNCTALGLALNV"
            "PGSSSHDTLCTSCTGFPLSTRV"
        ),
        "target": "TL1A",
        "note": "Natural decoy receptor ectodomain (PDB 3K51, TL1A trimer probe)"
    }
]

# Giới hạn số candidate chạy (None = toàn bộ, hoặc đặt số nguyên vd: 20 hoặc 50)
MAX_CANDIDATES = None

# Bộ lọc Target: None = chạy toàn bộ; hoặc đặt 'TL1A' / 'IL7RA' để chỉ chạy riêng target đó
TARGET_FILTER = None

# Tự động tìm kiếm file candidate_pool_for_screening.csv trên Drive / content / repo
candidate_search_paths = [
    INPUT_CANDIDATES_PATH,
    f"{D_OUT}/candidate_pool_for_screening.csv",
    f"{D_DATA}/candidate_pool_for_screening.csv",
    f"{ROOT}/candidate_pool_for_screening.csv",
    "/content/candidate_pool_for_screening.csv",
    "/content/proteinAI/notebooks/candidate_pool_for_screening.csv",
    "candidate_pool_for_screening.csv",
    "notebooks/candidate_pool_for_screening.csv",
]
CAND_FILE_FOUND = next((p for p in candidate_search_paths if p and os.path.exists(p)), None)

if CAND_FILE_FOUND:
    CANDIDATES = parse_candidates_file(CAND_FILE_FOUND)
    print(f"Nạp thành công {len(CANDIDATES)} candidate từ: {CAND_FILE_FOUND}")
else:
    CANDIDATES = SAMPLE_BENCHMARK_CANDIDATES
    print(f"Sử dụng tập Benchmark Đối chứng chuẩn ({len(CANDIDATES)} sequences) để kiểm định pipeline.")

if TARGET_FILTER:
    before_n = len(CANDIDATES)
    CANDIDATES = [c for c in CANDIDATES if c.get("target", "").upper() == TARGET_FILTER.upper()]
    print(f"Lọc theo TARGET_FILTER='{TARGET_FILTER}': {len(CANDIDATES)} / {before_n} candidate(s)")

if MAX_CANDIDATES and len(CANDIDATES) > MAX_CANDIDATES:
    print(f"Giới hạn số candidates sàng lọc theo MAX_CANDIDATES: {MAX_CANDIDATES} / {len(CANDIDATES)}")
    CANDIDATES = CANDIDATES[:MAX_CANDIDATES]

# Kiểm tra tính hợp lệ của ký tự amino acid
for c in CANDIDATES:
    bad = sorted(set(c["sequence"]) - VALID_AA)
    if bad:
        raise ValueError(f"Candidate '{c['name']}' chứa ký tự không hợp lệ: {bad}")

print("\n" + "-"*88)
print(f"{'#':<3} {'Candidate Name':<32} {'Target':<8} {'Length':>6}  {'Ghi chú / Nguồn'}")
print("-"*88)
for i, c in enumerate(CANDIDATES, 1):
    print(f"{i:<3} {c['name']:<32} {c['target']:<8} {len(c['sequence']):>6d}  {c.get('note', '')[:40]}")
print("-"*88)

REPORT["candidates_input_count"] = len(CANDIDATES)
budget("(cell 8 xong)")

## Cell 9 — VÒNG LẶP SÀNG LỌC ESMFOLD2 + ipSAE + NOVELTY CHECK

Thực hiện folding từng phức hợp bằng ESMFold2 (1 seed, 1 sample, no template), tính toán ipSAE giao diện và kiểm tra tính mới theo thời gian thực.

In [ ]:
# ============================== CELL 9: SCREENING LOOP ==========================
budget("(bắt đầu cell 9: Vòng lặp sàng lọc)")
import warnings

num_loops  = 10 if FAST_MODE else GATE_NUM_LOOPS
num_steps  = 50 if FAST_MODE else GATE_SAMPLING_STEPS
lm_dropout = GATE_LM_DROPOUT

SCREENED_RESULTS = []

completed_by_name = {}
csv_all_check = f"{D_OUT}/03_screening_results.csv"
if os.path.exists(csv_all_check):
    try:
        with open(csv_all_check, "r", encoding="utf-8") as fh:
            for row in csv.DictReader(fh):
                nm = row.get("name")
                if nm and row.get("screen_verdict"):
                    completed_by_name[nm] = {
                        "name": nm, "target": row.get("target"),
                        "length": int(row["length"]) if row.get("length") else None,
                        "screen_verdict": row.get("screen_verdict"),
                        "ipsae_binder": float(row["ipsae_binder"]) if row.get("ipsae_binder") else 0.0,
                        "ipsae_min": float(row["ipsae_min"]) if row.get("ipsae_min") else 0.0,
                        "iptm": float(row["iptm"]) if row.get("iptm") else None,
                        "ptm": float(row["ptm"]) if row.get("ptm") else None,
                        "plddt_mean": float(row["plddt_mean"]) if row.get("plddt_mean") else None,
                        "verdict": row.get("novelty_verdict", "PASS"),
                        "max_identity_pct": float(row["max_identity_pct"]) if row.get("max_identity_pct") else 0.0,
                        "coverage_ours_pct": float(row["coverage_ours_pct"]) if row.get("coverage_ours_pct") else 0.0,
                        "worst_hit_id": row.get("worst_hit_id", ""),
                        "fold_seconds": float(row["fold_seconds"]) if row.get("fold_seconds") else 0.0,
                        "vram_peak_gb": float(row["vram_peak_gb"]) if row.get("vram_peak_gb") else 0.0,
                        "ok": True,
                    }
    except Exception as e:
        print(f"Lưu ý khi đọc checkpoint CSV cũ: {e}")

if os.path.exists(PARTIAL):
    try:
        with open(PARTIAL, "r", encoding="utf-8") as fh:
            prev_p = json.load(fh)
            for rec in prev_p.get("screened_candidates", []):
                if rec.get("ok") and rec.get("name"):
                    completed_by_name[rec["name"]] = rec
    except Exception as e:
        print(f"Lưu ý khi đọc checkpoint partial JSON cũ: {e}")

if completed_by_name:
    print(f"\n[CHECKPOINT RESUME] Tìm thấy {len(completed_by_name)} candidate(s) đã hoàn tất từ phiên trước trên Drive!")
    print("Các candidate này sẽ được BỎ QUA FOLD và giữ nguyên kết quả đo lường để tiết kiệm compute.\n")

print(f"BẮT ĐẦU SÀNG LỌC {len(CANDIDATES)} CANDIDATE(S)")
print(f"Cấu hình fold: loops={num_loops} · steps={num_steps} · seed=0 · sample=1 · no-template\n")

for idx, cand in enumerate(CANDIDATES, 1):
    c_name = cand["name"]
    c_seq  = cand["sequence"]
    c_tgt  = cand.get("target", "IL7RA").upper()
    c_tag  = f"screen_{idx:03d}_{c_name[:24]}"
    outdir = f"{D_RUNS}/{c_tag}"
    
    # Nếu đã có trong checkpoint, nạp lại kết quả và bỏ qua fold
    if c_name in completed_by_name:
        old_rec = completed_by_name[c_name]
        print(f"[{idx:>3}/{len(CANDIDATES)}] ĐÃ CÓ (Resume): {c_name:<30} -> ipSAE={old_rec.get('ipsae_binder', 0.0):.4f} · {old_rec.get('screen_verdict')} (Bỏ qua fold)")
        SCREENED_RESULTS.append(old_rec)
        REPORT["screened_candidates"] = SCREENED_RESULTS
        continue
    
    est_time_s = 40 if c_tgt == "IL7RA" else 110
    if not have_budget(est_time_s / 60.0 + 0.5, f"candidate {idx}: {c_name}"):
        print(f"[CẢNH BÁO] Hết ngân sách thời gian trước candidate {idx}. Dừng sàng lọc an toàn.")
        break

    print(f"\n{'='*80}")
    print(f"[{idx}/{len(CANDIDATES)}] Sàng lọc: {c_name} (Target: {c_tgt}, Len: {len(c_seq)} aa)")
    print(f"{'='*80}")
    
    # 1. Xây dựng phức hợp theo target
    if c_tgt == "IL7RA":
        # Monomer IL-7Ra (Chain A) + Candidate (Chain B)
        chains = [("A", IL7RA_SEQ, MSA_IL7RA), ("B", c_seq, None)]
        binder_chain = "B"
        total_res = len(IL7RA_SEQ) + len(c_seq)
    elif c_tgt == "TL1A":
        # Trimer TL1A (Chains A, B, C broadcast MSA) + Candidate (Chain Z)
        chains = [(["A", "B", "C"], TL1A_SEQ, MSA_TL1A), ("Z", c_seq, None)]
        binder_chain = "Z"
        total_res = 3 * len(TL1A_SEQ) + len(c_seq)
    else:
        print(f"  [LỖI] Target không nhận diện: {c_tgt}. Bỏ qua.")
        continue
        
    spi = StructurePredictionInput(sequences=[
        ProteinInput(id=cid, sequence=seq, msa=msa) for cid, seq, msa in chains
    ])
    
    # 2. Thực hiện fold trên GPU
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t_start = time.time()
    fold_ok = False
    rec = {
        "name": c_name, "target": c_tgt, "length": len(c_seq), "total_complex_residues": total_res,
        "tag": c_tag, "binder_chain": binder_chain, "ok": False
    }
    
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result = builder.fold(
                model, spi, num_loops=num_loops, num_sampling_steps=num_steps,
                num_diffusion_samples=1, seed=0, lm_dropout=lm_dropout,
                msa_max_depth=MSA_MAX_DEPTH, msa_column_mask_rate=MSA_COLUMN_MASK_RATE,
                complex_id=c_tag
            )
        fold_s = time.time() - t_start
        peak_vram = torch.cuda.max_memory_allocated() / 1024**3
        
        rec["fold_seconds"] = round(fold_s, 1)
        rec["vram_peak_gb"] = round(peak_vram, 2)
        rec["ptm"]  = round(float(result.ptm), 4) if result.ptm is not None else None
        rec["iptm"] = round(float(result.iptm), 4) if result.iptm is not None else None
        
        if result.plddt is not None:
            pl = result.plddt.detach().cpu().float().numpy().ravel()
            rec["plddt_mean"] = round(float(pl.mean() * (100.0 if pl.max() <= 1.0 else 1.0)), 2)
            
        # 3. Chạy ipSAE adapter
        ipsae_res = run_ipsae_eval(result, c_tag, outdir, binder_chain=binder_chain)
        rec["ipsae_binder"] = ipsae_res.get("ipsae_binder_best", 0.0)
        rec["ipsae_min"]    = ipsae_res.get("ipsae_min", 0.0)
        rec["ipsae_pairs"]  = ipsae_res.get("binder_asym_pairs", {})
        
        # 4. Kiểm tra tính mới (Novelty Check)
        known_target_db = KB_DB.get(c_tgt, [])
        nov_res = evaluate_novelty(c_name, c_seq, known_target_db)
        rec.update(nov_res)
        
        # 5. Phán quyết sàng lọc tổng hợp
        # Tiêu chí: Novelty == PASS VÀ ipSAE ≥ 0.60 (ngưỡng mạnh) hoặc xem xét xếp hạng
        rec["ok"] = True
        rec["screen_verdict"] = "PASS" if (rec["verdict"] == "PASS" and rec["ipsae_binder"] >= 0.50) else "REVIEW"
        if rec["verdict"] == "FAIL":
            rec["screen_verdict"] = "FAIL_NOVELTY"
            
        print(f"  Fold: {fold_s:.1f}s · VRAM: {peak_vram:.2f} GB · pLDDT: {rec.get('plddt_mean')} · ipTM: {rec['iptm']}")
        print(f"  ipSAE(binder): {rec['ipsae_binder']:.4f} · Cặp giao diện: {rec['ipsae_pairs']}")
        print(f"  Novelty: {rec['verdict']} (MaxID: {rec['max_identity_pct']:.1f}%, Cov: {rec['coverage_ours_pct']:.1f}%) -> {rec['screen_verdict']}")
        
        del result
    except torch.cuda.OutOfMemoryError as e:
        rec["error"] = "CUDA_OOM"
        print(f"  [LỖI] OOM ở {total_res} residues: {e}")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"
        print(f"  [LỖI] {rec['error']}")
    finally:
        gc.collect(); torch.cuda.empty_cache()
        
    SCREENED_RESULTS.append(rec)
    REPORT["screened_candidates"] = SCREENED_RESULTS
    save_partial()

print(f"\nHoàn tất sàng lọc {len(SCREENED_RESULTS)} candidate(s).")
budget("(cell 9 xong)")

## Cell 10 — XUẤT KẾT QUẢ & BÁO CÁO (PASTE CHO CLAUDE)

Xếp hạng candidates theo điểm ipSAE tương tác, lọc theo tiêu chí tính mới, xuất file CSV submit + JSON báo cáo trên Google Drive và in khối `PASTE CHO CLAUDE`.

In [ ]:
# ============================== CELL 10: EXPORT & REPORT ========================
budget("(bắt đầu cell 10: Xuất báo cáo)")

# 1. Sắp xếp candidates: Ưu tiên Novelty PASS, sau đó ipSAE giảm dần, sau đó ipTM giảm dần
def sort_key(r):
    verdict_rank = 0 if r.get("verdict") == "PASS" else 1
    ipsae = r.get("ipsae_binder") or 0.0
    iptm  = r.get("iptm") or 0.0
    return (verdict_rank, -ipsae, -iptm)

sorted_candidates = sorted(SCREENED_RESULTS, key=sort_key)

# 2. Ghi bảng kết quả tổng hợp ra CSV
CSV_ALL_PATH = f"{D_OUT}/03_screening_results.csv"
CSV_TOP_PATH = f"{D_OUT}/03_screening_top_candidates.csv"
JSON_REP_PATH = f"{D_OUT}/03_screening_report.json"

CSV_FIELDS = [
    "rank", "name", "target", "length", "screen_verdict",
    "ipsae_binder", "ipsae_min", "iptm", "ptm", "plddt_mean",
    "novelty_verdict", "max_identity_pct", "coverage_ours_pct", "worst_hit_id",
    "fold_seconds", "vram_peak_gb", "error"
]

rows_to_write = []
for rk, c in enumerate(sorted_candidates, 1):
    row = {
        "rank": rk,
        "name": c.get("name"),
        "target": c.get("target"),
        "length": c.get("length"),
        "screen_verdict": c.get("screen_verdict"),
        "ipsae_binder": c.get("ipsae_binder"),
        "ipsae_min": c.get("ipsae_min"),
        "iptm": c.get("iptm"),
        "ptm": c.get("ptm"),
        "plddt_mean": c.get("plddt_mean"),
        "novelty_verdict": c.get("verdict"),
        "max_identity_pct": c.get("max_identity_pct"),
        "coverage_ours_pct": c.get("coverage_ours_pct"),
        "worst_hit_id": c.get("worst_hit_id"),
        "fold_seconds": c.get("fold_seconds"),
        "vram_peak_gb": c.get("vram_peak_gb"),
        "error": c.get("error", ""),
    }
    rows_to_write.append(row)

with open(CSV_ALL_PATH, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=CSV_FIELDS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows_to_write)
print(f"Bảng kết quả toàn bộ: {CSV_ALL_PATH}")

# Top candidates (chỉ lấy Novelty PASS)
top_candidates = [r for r in rows_to_write if r.get("novelty_verdict") == "PASS"]
with open(CSV_TOP_PATH, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=CSV_FIELDS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(top_candidates[:20])
print(f"Bảng Top candidates (PASS novelty): {CSV_TOP_PATH}")

for tgt_key in ("IL7RA", "TL1A"):
    tgt_cands = [r for r in top_candidates if r.get("target", "").upper() == tgt_key][:20]
    if tgt_cands:
        tgt_csv_path = f"{D_OUT}/03_screening_top_{tgt_key}.csv"
        with open(tgt_csv_path, "w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=CSV_FIELDS, extrasaction="ignore")
            w.writeheader()
            w.writerows(tgt_cands)
        print(f"Bảng Top 20 {tgt_key} (PASS novelty): {tgt_csv_path}")

# 3. Lưu JSON báo cáo đầy đủ
REPORT["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
REPORT["elapsed_min"]  = round(elapsed_min(), 2)
REPORT["cu_estimate"]  = round(cu_burned(), 3)
REPORT["summary_table"] = rows_to_write

with open(JSON_REP_PATH, "w", encoding="utf-8") as fh:
    json.dump(REPORT, fh, indent=2, default=str)
print(f"Báo cáo JSON: {JSON_REP_PATH}")

# 4. In bảng tóm tắt định dạng ASCII ra console
print("\n" + "="*96)
print(f"{'Rk':<3} {'Candidate':<26} {'Tgt':<6} {'Len':>4} {'ipSAE':>8} {'ipTM':>6} {'pLDDT':>6} {'Novelty':<8} {'MaxID%':>7} {'Verdict'}")
print("="*96)
for r in rows_to_write:
    ipsae_str = f"{r['ipsae_binder']:.4f}" if r.get('ipsae_binder') is not None else "N/A"
    iptm_str  = f"{r['iptm']:.3f}" if r.get('iptm') is not None else "N/A"
    plddt_str = f"{r['plddt_mean']:.1f}" if r.get('plddt_mean') is not None else "N/A"
    id_str    = f"{r['max_identity_pct']:.1f}%" if r.get('max_identity_pct') is not None else "N/A"
    print(f"{r['rank']:<3} {r['name'][:25]:<26} {r['target']:<6} {r['length']:>4} "
          f"{ipsae_str:>8} {iptm_str:>6} {plddt_str:>6} {str(r['novelty_verdict']):<8} {id_str:>7} {r['screen_verdict']}")
print("="*96)

# 5. In khối báo cáo chuẩn PASTE CHO CLAUDE
print("\n========================================")
print("PASTE CHO CLAUDE — BẮT ĐẦU")
print("========================================")
print(json.dumps({
    "notebook": "03_esmfold2_screening",
    "elapsed_min": REPORT["elapsed_min"],
    "cu_estimate": REPORT["cu_estimate"],
    "candidates_screened": len(SCREENED_RESULTS),
    "top_candidates_count": len(top_candidates),
    "top_5": rows_to_write[:5],
    "files": {
        "csv_all": CSV_ALL_PATH,
        "csv_top": CSV_TOP_PATH,
        "json_report": JSON_REP_PATH
    }
}, indent=2))
print("========================================")
print("PASTE CHO CLAUDE — KẾT THÚC")
print("========================================")
budget("(toàn bộ pipeline hoàn tất)")